# 01 - Exploratory Data Analysis

Quick EDA of the Peacock subscriber sample: net adds by tier, tentpole
seasonality, price changes, and usage per paid sub. Run from the project
root with the project virtual environment.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
from src.utils import load_config
from src.data_loader import load_raw_data, add_series_id

config = load_config(os.path.abspath('../config/config.yaml'))
df = add_series_id(load_raw_data(config, os.path.abspath('../data/sample_input.csv')), config)
df['net_adds'] = df['gross_adds'] - df['churned_subs']
print(df.shape)
df.head()

In [ ]:
# Grain and coverage
print('Segments:', df['series_id'].nunique())
print('Tiers:', df['tier'].nunique(), '| Channels:', df['acquisition_channel'].nunique())
print('Date range:', df['date'].min(), '->', df['date'].max())
df.groupby('tier')[['gross_adds', 'churned_subs', 'net_adds']].sum()

In [ ]:
# Paid subscribers over time by tier
subs = df.groupby(['date', 'tier'])['paid_subs_eod'].sum().unstack()
ax = subs.plot(figsize=(11, 4), title='Paid subscribers by tier')
ax.set_ylabel('Paid subs'); plt.show()

In [ ]:
# Tentpole vs non-tentpole gross adds and churn
df.groupby('tentpole_flag')[['gross_adds', 'churned_subs']].mean()

In [ ]:
# Biggest tentpoles by gross adds
(df[df['tentpole_flag'] == 1]
   .groupby('tentpole_name')['gross_adds'].sum()
   .sort_values(ascending=False).head(10))

In [ ]:
# Churn vs list price for one segment (price increases show as churn bumps)
sid = df['series_id'].iloc[0]
one = df[df['series_id'] == sid]
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(one['date'], one['churned_subs'], label='churn', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(one['date'], one['list_price'], label='list price', color='tab:red', alpha=0.6)
ax1.set_title(f'Churn vs list price: {sid}'); plt.show()

In [ ]:
# Usage per paid sub (hours per sub per day) by tier
usage = df.groupby(['date', 'tier'])[['hours_watched', 'paid_subs_bod']].sum()
hps = (usage['hours_watched'] / usage['paid_subs_bod']).unstack()
hps.rolling(7).mean().plot(figsize=(11, 4), title='Hours per paid sub per day (7-day mean)')
plt.show()

Next: run `python main.py` from the project root to validate, engineer
features, train/select a model per target (gross adds, churn, hours),
forecast, and generate segment scenarios and the Growth OKR summary.